In [1]:
import pandas as pd
from dotenv import load_dotenv
import os
load_dotenv()

True

## Setting up PINs comparison

In [2]:
hpd_df = pd.read_parquet("data/hpd_violations.parquet")

In [3]:
nycha_df = pd.read_csv("data/nycha.csv")

In [9]:
nycha_pdi = nycha_df['Primary Building Identifier'].dropna().unique().tolist()
nycha_pdi = [int(x) for x in nycha_pdi]
len(nycha_pdi)


227

In [ ]:
nycha_df.head()

In [ ]:
hpd_df.head()

In [28]:
hpd_building_ids = hpd_df['BuildingID'].dropna().unique()
hpd_building_ids = [int(x) for x in hpd_building_ids]
hpd_building_ids_set = set(hpd_building_ids)



In [29]:
# get percentage of NYCHA PINs in hpd_building_ids

exists = []
missing = []

for pin in nycha_pdi:
    if pin in hpd_building_ids:
        exists.append(pin)
    else:
        missing.append(pin)

len(missing)

154

In [30]:
matched_pids = set(nycha_pdi) & set(hpd_building_ids)

In [31]:
len(matched_pids)

#still quite a few missing

73

## Setting up BINs comparison

In [20]:
nycha_bins = nycha_df['BIN'].dropna().unique().tolist()
nycha_bins = [int(x) for x in nycha_bins]
len(nycha_bins)


226

In [32]:
# BIN in nycha is float64 (e.g. 1234567.0), convert to nullable int then string
nycha_df["BIN_key"] = nycha_df["BIN"].astype("Int64").astype(str)

# BIN in large dataset is object — strip any whitespace/decimals just in case
hpd_df["BIN_key"] = hpd_df["BIN"].astype("Int64").astype(str).str.strip()

In [33]:
small_bins = set(nycha_df["BIN_key"].dropna())
large_bins = set(hpd_df["BIN_key"].dropna())

matched_bins = small_bins & large_bins
print(f"NYCHA BINs:         {len(small_bins)}")
print(f"Matched in large:   {len(matched_bins)}")
print(f"Match rate:         {len(matched_bins)/len(small_bins)*100:.1f}%")

NYCHA BINs:         227
Matched in large:   73
Match rate:         32.2%


In [38]:
matched_pids = {str(x) for x in matched_pids}

In [ ]:
# How many NYCHA records have either a matched PID or matched BIN

# All PIDs and BINs will be used its just a question of how much of NYCHA that covers

In [41]:
matched_nycha_records = nycha_df[
    nycha_df['BIN_key'].isin(matched_bins) | 
    nycha_df['Primary Building Identifier'].isin(matched_pids)
]

print(f"Matched records: {len(matched_nycha_records)} / {len(nycha_df)} ({len(matched_nycha_records)/len(nycha_df)*100:.1f}%)")

Matched records: 785 / 2391 (32.8%)


In [50]:
hpd_df.tail()

,ViolationID,BuildingID,RegistrationID,BoroID,Borough,HouseNumber,LowHouseNumber,HighHouseNumber,StreetName,StreetCode,...,RentImpairing,Latitude,Longitude,CommunityBoard,CouncilDistrict,CensusTract,BIN,BBL,NTA,BIN_key
10747002,9984323,52028,201562,2,BRONX,2700,2700,2736,BRONX PARK EAST,15220,...,N,40.865552,-73.870416,11.0,12.0,33601.0,2093450.0,2.045060e+09,Allerton,2093450
10747003,9984324,52028,201562,2,BRONX,2700,2700,2736,BRONX PARK EAST,15220,...,N,40.865552,-73.870416,11.0,12.0,33601.0,2093450.0,2.045060e+09,Allerton,2093450
10747004,9984325,52028,201562,2,BRONX,2700,2700,2736,BRONX PARK EAST,15220,...,N,40.865552,-73.870416,11.0,12.0,33601.0,2093450.0,2.045060e+09,Allerton,2093450
10747005,9984326,52028,201562,2,BRONX,2700,2700,2736,BRONX PARK EAST,15220,...,N,40.865552,-73.870416,11.0,12.0,33601.0,2093450.0,2.045060e+09,Allerton,2093450
10747006,9984327,52028,201562,2,BRONX,2700,2700,2736,BRONX PARK EAST,15220,...,N,40.865552,-73.870416,11.0,12.0,33601.0,2093450.0,2.045060e+09,Allerton,2093450


In [45]:
nycha_df.head()

,Violation ID,Primary Building Identifier,Primary Boro Identifier,Primary Borough Name,Primary House Number,Primary Low House Number,Primary High House Number,Primary Street Name,Primary Postcode,Development Name,...,Issued in Error,Latitude,Longitude,Community Board,Council District,BIN,BBL,Census Tract (2020),Neighborhood Tabulation Area (NTA) (2020),BIN_key
0,18220152,808768,3,BROOKLYN,1,1,17,LORRAINE STREET,11231,RED HOOK WEST,...,N,40.675118,-74.009568,306,38,3326207.0,3.005330e+09,85,BK0601,3326207
1,18220153,808768,3,BROOKLYN,1,1,17,LORRAINE STREET,11231,RED HOOK WEST,...,N,40.675118,-74.009568,306,38,3326207.0,3.005330e+09,85,BK0601,3326207
2,18220154,808768,3,BROOKLYN,1,1,17,LORRAINE STREET,11231,RED HOOK WEST,...,N,40.675118,-74.009568,306,38,3326207.0,3.005330e+09,85,BK0601,3326207
3,18223815,286990,3,BROOKLYN,621,615,621,EAST 105 STREET,11236,BREUKELEN,...,N,40.650186,-73.897621,318,42,3321415.0,3.081740e+09,982,BK1803,3321415
4,18223816,286990,3,BROOKLYN,621,615,621,EAST 105 STREET,11236,BREUKELEN,...,N,40.650186,-73.897621,318,42,3321415.0,3.081740e+09,982,BK1803,3321415
